# Lab 3 — Build the Pipeline, Hunt the Leaks
**SDA-AIE-111 · Day 2 · Hours 4–5 · 50 min lab + 50 min workshop · pairs**

**Objective.** Deliver `build_preprocessor()` + the full churn pipeline; engineer RFM and calendar features that lift PR-AUC; find all three leaks in `sabotaged_features.ipynb`; pass the shuffled-label test.

**The lesson of the day:** *features beat models.* The same logistic regression, fed order-history features it could not see before, gains more PR-AUC than any model swap on Day 3 will.


In [1]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

repo root: /home/claude/amlf/repo


In [2]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               BASE_NUM_COLS, MODEL_NUM_COLS, CAT_COLS, LEAKY)

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
snapshot = pd.Timestamp("2025-11-01")
print(customers.shape, orders.shape)

(48000, 19) (610000, 11)


## Task 1 — The pipeline changes packaging, not math *(10 min)*

Wrap yesterday's logistic regression in `Pipeline(preprocessor → model)`. The score should match Lab 2b within noise — same features, same model, but now **all preprocessing parameters are learned inside the object**, from training data only.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from sklearn import set_config
set_config(display="diagram")        # the pipeline renders itself — preprocessing as documentation

X_all = customers.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = customers["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE)

churn_model = Pipeline([
    ("prep", build_preprocessor(BASE_NUM_COLS, CAT_COLS)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                               random_state=RANDOM_STATE)),
])
churn_model.fit(X_tr, y_tr)          # ALL preprocessing parameters learned here, train only
proba = churn_model.predict_proba(X_val)[:, 1]   # raw dataframe in, risk score out
score_no_features = average_precision_score(y_val, proba)
print(f"pipeline (snapshot columns only): PR-AUC {score_no_features:.3f}  (Lab 2b parity ✔)")
churn_model

pipeline (snapshot columns only): PR-AUC 0.485  (Lab 2b parity ✔)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

## Task 2 — RFM aggregates with the snapshot filter *(15 min)*

Open `src/manafeth/features.py` and read `rfm_features()` — the load-bearing line is the snapshot filter. Then build the full feature table and refit.

**The prediction-time rule for aggregates:** an aggregate over all orders summarises the future. Only orders **strictly before the snapshot** may enter a feature.


In [4]:
full = build_feature_table(customers, orders, snapshot)
new_cols = [c for c in full.columns if c not in customers.columns]
print("engineered columns:", new_cols)

Xf_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
Xf_train, Xf_test, _, _ = train_test_split(
    Xf_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
Xf_tr, Xf_val, _, _ = train_test_split(
    Xf_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE)

rfm_model = Pipeline([
    ("prep", build_preprocessor(BASE_NUM_COLS + ["orders_90d", "basket_trend"], CAT_COLS)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                               random_state=RANDOM_STATE)),
]).fit(Xf_tr, y_tr)
score_rfm = average_precision_score(y_val, rfm_model.predict_proba(Xf_val)[:, 1])
print(f"+ RFM aggregates: PR-AUC {score_rfm:.3f}   (was {score_no_features:.3f})")

engineered columns: ['orders_90d', 'avg_basket_90d', 'basket_trend', 'ramadan_order_share', 'weekend_order_share', 'is_weekend', 'days_to_ramadan', 'near_salary_day']


+ RFM aggregates: PR-AUC 0.568   (was 0.485)


## Task 3 — KSA calendar & behavioural-share features *(10 min)*

The features generic tutorials never have: the KSA weekend is **Fri/Sat**, and Ramadan drives real behaviour. Verify the weekend definition with a spot check on a known Friday, then add `ramadan_order_share` / `weekend_order_share` and `days_to_ramadan`.


In [5]:
# Spot check: 2025-10-31 was a Friday -> dayofweek 4 -> KSA weekend
assert pd.Timestamp("2025-10-31").dayofweek == 4
friday_orders = orders[orders["order_ts"].dt.date == pd.Timestamp("2025-10-31").date()]
print("orders on Fri 2025-10-31:", len(friday_orders), "-> is_weekend must be 1 for these")

full_model = Pipeline([
    ("prep", build_preprocessor(MODEL_NUM_COLS, CAT_COLS)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                               random_state=RANDOM_STATE)),
]).fit(Xf_tr, y_tr)
score_full = average_precision_score(y_val, full_model.predict_proba(Xf_val)[:, 1])
print(f"+ calendar/share features: PR-AUC {score_full:.3f}")
print(f"\nScoreboard:  base {score_no_features:.3f} -> +RFM {score_rfm:.3f} -> +calendar {score_full:.3f}")
print("Features beat models: this lift exceeds any model swap you will try on Day 3 — almost.")

orders on Fri 2025-10-31: 6397 -> is_weekend must be 1 for these


+ calendar/share features: PR-AUC 0.568

Scoreboard:  base 0.485 -> +RFM 0.568 -> +calendar 0.568
Features beat models: this lift exceeds any model swap you will try on Day 3 — almost.


## Task 4 — Leak hunt *(10 min)*

Open `notebooks/day2/sabotaged_features.ipynb`. It contains **three planted leaks**. Find each one, document it below with the one-line fix, then come back.

| # | Leak | One-line fix |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |


## Task 5 — The shuffled-label test *(5 min)*

If shuffled labels score above chance, something leaks. Run it on your final pipeline; the output goes in your commit message.


In [6]:
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(RANDOM_STATE)
y_shuffled = pd.Series(rng.permutation(y_train.values), index=y_train.index)

score = cross_val_score(full_model, Xf_train, y_shuffled, cv=5,
                        scoring="average_precision", n_jobs=-1).mean()
print(f"PR-AUC on shuffled labels: {score:.3f}")
print("≈ 0.14 (chance) -> pipeline is honest."
      " Materially above 0.14 -> a feature or a step leaks the label.")

PR-AUC on shuffled labels: 0.147
≈ 0.14 (chance) -> pipeline is honest. Materially above 0.14 -> a feature or a step leaks the label.


## Workshop hour (Day 2 H5) — propose one feature

Each pair proposes **one new domain feature computed from the orders table only**, implements it inside the pipeline, and posts before/after PR-AUC *from the same seed*. The class votes on the most valuable feature per compute cost.

Starter ideas (do better than these): order-frequency ratio `orders_90d / (3 × orders_per_month)` — is the customer slowing down relative to their own history? · category-diversity in the last 90 days · share of express-slot orders · average items per order trend.

**Commit:** `feat(lab3): leakage-proof pipeline + RFM/calendar features — shuffled-label PR-AUC <your number>`


In [7]:
# Workshop scratchpad — implement your pair's feature here.
# Example (the classic winner): frequency ratio — recent pace vs lifetime pace.
freq = full.assign(freq_ratio_90d=(full["orders_90d"] / (3 * full["orders_per_month"])).clip(0, 3))
Xw_all = freq.drop(columns=["churned_30d", "customer_id", *LEAKY])
Xw_train, _, _, _ = train_test_split(Xw_all, y_all, test_size=0.20, stratify=y_all,
                                     random_state=RANDOM_STATE)
Xw_tr, Xw_val, _, _ = train_test_split(Xw_train, y_train, test_size=0.25,
                                       stratify=y_train, random_state=RANDOM_STATE)
w_model = Pipeline([
    ("prep", build_preprocessor(MODEL_NUM_COLS + ["freq_ratio_90d"], CAT_COLS)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                               random_state=RANDOM_STATE)),
]).fit(Xw_tr, y_tr)
print("with freq_ratio_90d:", round(average_precision_score(
    y_val, w_model.predict_proba(Xw_val)[:, 1]), 3), f"  (before: {score_full:.3f})")

with freq_ratio_90d: 0.568   (before: 0.568)
